In [ ]:
import pandas as pd
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# Connect to the BioCascade database
con = duckdb.connect('../data/processed/biocascade.db')
df = con.execute("SELECT * FROM raw_patient_data").df()

print(f"✅ Data Loaded: {df.shape[0]} patients and {df.shape[1]} clinical variables.")
df.head()

In [ ]:
plt.figure(figsize=(14, 6))
# Creating a heatmap of missing values
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='magma')

plt.title('Clinical Data Coverage Audit: Identifying Gaps in 30+ Biomarkers', fontsize=15)
plt.xlabel('Medical Variables (Biomarkers)', fontsize=12)
plt.ylabel('Patient Records (N=5,533)', fontsize=12)
plt.show()

In [ ]:
# Selection of core biomarkers representing the three systems
cascade_features = [
    'age', 'systolic_bp', 'diastolic_bp', 'hba1c', 
    'fasting_glucose', 'triglycerides', 'hdl_cholesterol', 
    'serum_creatinine', 'bmi'
]

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(df[cascade_features].corr(), dtype=bool))
sns.heatmap(df[cascade_features].corr(), mask=mask, annot=True, cmap='coolwarm', fmt=".2f")

plt.title('Systemic Interaction Matrix: Evidence of BioCascade Connectivity', fontsize=15)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='age', bins=30, kde=True, color='#00f2ff')

plt.axvline(df['age'].mean(), color='red', linestyle='--', label=f'Mean Age: {df["age"].mean():.1f}')
plt.axvline(60, color='orange', linestyle=':', label='High-Risk Threshold (Observed)')

plt.title('Demographic Distribution: Targeting the Aging Cascade', fontsize=15)
plt.xlabel('Age (Years)')
plt.legend()
plt.show()

In [ ]:
# Statistical summary of the primary biomarkers
stats = df[['systolic_bp', 'diastolic_bp', 'hba1c', 'triglycerides', 'serum_creatinine']].describe()
print("📋 Clinical Range Audit:")
stats

In [ ]:
def create_table1(df):
    summary = {'Characteristic': [], 'Value': []}
    
    # Demographics
    summary['Characteristic'].append('Age (years), mean ± SD')
    summary['Value'].append(f"{df['age'].mean():.1f} ± {df['age'].std():.1f}")
    
    # Note: gender 2 is usually Female in NHANES
    female_count = (df['gender'] == 2).sum()
    female_pct = (female_count / len(df)) * 100
    summary['Characteristic'].append('Female, n (%)')
    summary['Value'].append(f"{female_count} ({female_pct:.1f}%)")
    
    # Biomarkers (Using your exact DB names)
    metrics = [
        ('BMI (kg/m²)', 'bmi'),
        ('HbA1c (%)', 'hba1c'),
        ('Systolic BP (mmHg)', 'systolic_bp'),
        ('Diastolic BP (mmHg)', 'diastolic_bp'),
        ('Serum Creatinine (mg/dL)', 'serum_creatinine')
    ]
    
    for label, col in metrics:
        if col in df.columns:
            summary['Characteristic'].append(f"{label}, mean ± SD")
            summary['Value'].append(f"{df[col].mean():.2f} ± {df[col].std():.2f}")

    return pd.DataFrame(summary)

print("📊 TABLE 1: Clinical Baseline")
print(create_table1(df).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Heatmap
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis', ax=axes[0])
axes[0].set_title("Missing Data Heatmap (Yellow = Missing)")

# Right: Bar chart of missingness %
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

missing_pct.plot(kind='barh', ax=axes[1], color='#ff6b6b')
axes[1].set_xlabel('% Missing')
axes[1].set_title('Biomarker Missingness %')
axes[1].axvline(x=30, color='red', linestyle='--', label='30% Inclusion Threshold')
axes[1].legend()

plt.tight_layout()
plt.show()